# Aula 3 — Lab da API exposta (CredSim)

Cliente HTTP que ataca o **próprio backend FastAPI** da CredSim — a API é, ela mesma, a superfície desta aula. **Pré-requisito:** app no ar — na raiz do projeto:

```
docker compose up --build
```

Financeira A em http://localhost:8000. Estrutura: **cenário negativo** (vulnerável) → **cenário positivo** (mitigado), com evidência nos logs.

In [ ]:
import os, requests
BASE = os.environ.get('CREDSIM_URL', 'http://localhost:8000')

def set_defenses(input_validation=False, output_validation=False, least_privilege=False, api_security=False):
    return requests.post(BASE + '/api/defenses', json={
        'input_validation': input_validation, 'output_validation': output_validation,
        'least_privilege': least_privilege, 'api_security': api_security,
    }).json()

def get_conversa(conversa_id, solicitante):
    return requests.get(BASE + f'/api/conversas/{conversa_id}', params={'solicitante': solicitante}).json()

def chamar_publica(cliente_id, pergunta):
    return requests.post(BASE + '/api/publica', json={'cliente_id': cliente_id, 'pergunta': pergunta}).json()

def reset():
    requests.post(BASE + '/api/reset')

def mostrar_logs(n=8):
    for e in requests.get(BASE + '/api/logs').json()[:n]:
        flags = []
        if e.get('stage') == 'idor': flags.append('autorizado=' + str(e.get('autorizado')))
        if e.get('stage') == 'rate_limit': flags.append('BLOQUEADO POR RATE LIMIT')
        print('#' + str(e['seq']) + ' [' + e['stage'] + '] ' + (' '.join(flags) or '-'))

try:
    print('Conectado:', requests.get(BASE + '/api/info', timeout=3).json())
except Exception as e:
    print('App não respondeu — rode `docker compose up --build` na raiz.'); print(e)

## ❌ Cenário NEGATIVO — IDOR (ID de conversa previsível)
O endpoint devolve o histórico de conversa por um ID sequencial, sem checar de quem é a conversa. O cliente-A troca o número na URL e recebe dado do cliente-B — o próprio token era válido, só não deveria dar acesso àquele recurso (**IDOR clássico do appsec, agora sobre LLM02**).

In [ ]:
reset()
set_defenses()
r = get_conversa(2, solicitante='cliente-A')  # conversa 2 é do cliente-B
print('Status:', r['status'], '| dono real:', r['dono_real'], '| autorizado?', r['autorizado'])
if r['autorizado']:
    print('Vazou:', r['cliente_nome'], '-', r['cpf'], '-', r['resumo'])
print('\n--- evidência nos logs ---'); mostrar_logs(2)

## ❌ Cenário NEGATIVO — sem rate limit (LLM10)
Um parceiro externo chama a API pública sem limite algum — o custo cresce sem parar (denial of wallet).

In [ ]:
ultimo = None
for _ in range(8):
    ultimo = chamar_publica('parceiro-x', 'qual a taxa de hoje?')
print('Bloqueado?', ultimo['bloqueado'], '| chamada nº', ultimo['chamada_numero'], '| custo total: US$', ultimo['custo_total_usd'])
print('\n--- evidência nos logs ---'); mostrar_logs(3)

## ✅ Cenário POSITIVO — autorização por recurso + rate limit
**Mitigação:** `api_security` ON valida que quem pede é o **dono do recurso** (não só "está autenticado") e aplica rate limit por cliente.

In [ ]:
reset()
set_defenses(api_security=True)

r = get_conversa(2, solicitante='cliente-A')
print('IDOR bloqueado? status', r['status'], '| autorizado?', r['autorizado'])

ultimo = None
for _ in range(8):
    ultimo = chamar_publica('parceiro-x', 'qual a taxa de hoje?')
print('Rate limit bloqueou? status', ultimo['status'], '| bloqueado?', ultimo['bloqueado'])
print('\n--- evidência nos logs ---'); mostrar_logs(4)

## Conclusão
- **Negativo:** "está autenticado" não é o mesmo que "está autorizado" (IDOR); e sem rate limit o custo — e o abuso — não têm teto (LLM10).
- **Positivo:** validar o dono do recurso + limitar chamadas por cliente contém os dois vetores sem impedir o uso legítimo.
- Encerra a Aula 3: a superfície de ataque é a pilha inteira (Aula 1) — cada arquitetura acende uma camada nova. As defesas a fundo, na **Aula 5**; dados e privacidade, na **Aula 4**.